In [2]:
# ========== 环境检查 + 导入：确认解释器，并搬进本课常用工具 ==========

# 打印当前 Jupyter 正在用的 Python 可执行文件路径（排查装错环境时很有用）
import sys; print(sys.executable)

# 导入标准库 os：读环境变量（Environment Variables）等
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：可对接云端，也可把 base_url 指到本地 Ollama
from openai import OpenAI
# 如果运行此单元时出现错误，请转到故障排除笔记本！


c:\Users\Walja\Downloads\AI learning\side project\.venv\Scripts\python.exe


#### 连通性测试

先确认本机 **Ollama**（OpenAI 兼容接口）能正常回答；通过后再做网站摘要等练习。


In [3]:
# ========== 冒烟测试：用 OpenAI 兼容客户端打本地 Ollama ==========

# 导入 OpenAI 客户端（即使连的是本地 Ollama，也走同一套 chat.completions API）
from openai import OpenAI

# base_url 指向本机 Ollama 的 OpenAI 兼容端点 /v1；api_key 对本地常填占位字符串 'ollama'
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# 发起一次 Chat Completions：model 必须是本机已 pull 的名字
response = ollama.chat.completions.create(
    model="gemma3:4b",
    messages=[
        # system：设定回答风格（发给模型的英文指令保持原样）
        {"role":"system","content":"you a formal chinese speaker"},
        # user：让模型确认可用，并写下「开始学习 AI / LLM 工程」相关句子
        {"role": "user", "content": "Hello, confirm you're working., write down that we are starting to learning AI engineering or llm engineering "}]
)
# 从 choices[0].message.content 取出助手回复文本并打印
print(response.choices[0].message.content)


好的，我确认我在工作中。

我们现在正在开始学习人工智能工程或大型语言模型（LLM）工程的学习。 (Hǎo de, wǒ quèrén wǒmen zài gōngzuò zhōng. Wǒmen xiànzài zhèngzài kāishǐ xuéxí rénlishi gengyòng yuánchéng huò dàxíng yǔyán móxìng (LLM) yīnggong de xuéxí.)

(Okay, I confirm that I’m working. We are now starting to begin learning AI engineering or LLM engineering.) 

Do you have a specific question about this topic you'd like me to address?


#### 测试通过

若上一格已正常打印中文回复，说明本地 **Ollama + gemma3:4b** 通路可用，可以继续往下做项目。


In [ ]:
# ========== 认识 messages：最简单的 user 消息结构 ==========

# 用户要发给模型的一句话（字符串内容保持英文原文）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 期望的列表：每项含 role 与 content
messages = [{"role": "user", "content": message}]

# 在笔记本里单独写变量名：会展示该对象，便于检查结构
messages


In [42]:
# ========== 再测一次：只有 user、没有 system 的最简调用 ==========

from openai import OpenAI

# 同样连接到本地 Ollama 的 OpenAI 兼容接口
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# messages 里只有一条 user；对比上一格「带 system」时口吻可能不同
response = ollama.chat.completions.create(
    model="gemma3:4b",
    messages=[{"role": "user", "content": "Hello, confirm you're working., write down that we are starting to learning AI engineering or llm engineering "}]
)

# 打印模型回复
print(response.choices[0].message.content)


Okay! Yes, I’m working.

Let’s start by confirming: **We are beginning our journey into the exciting world of AI Engineering and LLM (Large Language Model) Engineering!** 🎉 

I'm here to help you every step of the way. Let's learn together. 😊



## 开始我们的第一个项目

目标：抓取/汇总网页（或新闻）内容，用本地模型生成**简短、带点吐槽风格**的 Markdown 摘要——练习 `system` / `user` prompt 与 `messages` 组装。


In [4]:
# ========== 定义 system prompt：规定助手人设与输出格式 ==========
# 可稍后实验：把最后一句改成「用西班牙语以 markdown 方式回复」等
# 发给模型的英文指令保持原样（翻译会改变行为）

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [5]:
# ========== 定义 user prompt 前缀：告诉模型「下面是网页正文，请摘要」==========

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## Messages（对话消息结构）

OpenAI 的 API 期望接收特定结构的消息；许多其他兼容 API（包括 Ollama 的 `/v1`）也共享这一结构：

```python
[
    {"role": "system", "content": "此处显示系统消息"},
    {"role": "user", "content": "用户消息在此处"}
]
```

下面 2 个单元格先做一个很简单的调用预览——我们还不会立刻上最强的云端 GPT。


In [9]:
# ========== 预览调用：system + user 的最小可运行例子 ==========

# 组装 messages：system 定人设，user 提一个简单算术问题
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 连接本地 Ollama（OpenAI 兼容）
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# 用 gemma3:4b 回答；参数与云端 Chat Completions 同形
response = ollama.chat.completions.create(model="gemma3:4b", messages=messages)

# 打印助手回复
print(response.choices[0].message.content)


2 + 2 = 4! 😊 

Do you want to try another math problem?


## 用函数组装给模型的 messages

把 `system_prompt` 与 `user_prompt_prefix + 网页正文` 打包成标准 `messages` 列表，避免每次手写重复结构。


In [7]:
# ========== 辅助函数：把 website 文本打成 Chat Completions 所需 messages ==========

def messages_for(website):
    # 返回与上面演示完全相同的结构：system + user
    return [
        {"role": "system", "content": system_prompt},
        # user：前缀说明任务，再拼接具体网站正文/来源字符串
        {"role": "user", "content": user_prompt_prefix + website}
    ]


### 注意：下面示例是「虚构且不准确」的

本地 **Ollama 默认没有外网**，模型并不能真正去抓 BBC/CNN 实时新闻；它只会根据提示词「编」一份看起来像新闻分析的内容。要做真文章，请用后面 `newspaper` / `requests` 抓取单元格。


In [10]:
# ========== 演示：用 messages_for 调本地模型做「假想」新闻分析 ==========

from openai import OpenAI

# 定义提示（发给模型的英文保持原样；可换成你自己的任务描述）
system_prompt = "You are a senior news analyst."
user_prompt_prefix = """Retrieve and analyze the top news stories for today, July 11, 2026, from leading international news sources. Provide a summary and impact analysis for the top 5 stories. Format as specified."""

def messages_for(website):
    # system 定分析师人设；user = 任务前缀 + 来源说明
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + " Source: " + website}
    ]

# 连接到本地 Ollama 的 OpenAI 兼容端点
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# 选择一个「来源」字符串（模型并无真实联网检索）
source = "BBC, CNN, Reuters, Al Jazeera, AP"

# 得到回应：create 返回 ChatCompletion；内容在 choices[0].message.content
response = ollama.chat.completions.create(
    model="gemma3:4b",
    messages=messages_for(source)
)
# 用 Markdown 在笔记本里渲染助手回复（比纯 print 更好读）
display(Markdown(response.choices[0].message.content))
# # 打印结果
# 打印（响应.选择[0].消息.内容）


Okay, here’s an analysis of the top five news stories for July 11th, 2026, compiled from BBC, CNN, Reuters, Al Jazeera, and Associated Press reports.  This is a simulated report based on extrapolations of current trends and projected developments – essentially creating a plausible near-future scenario. I’ve aimed to provide the depth and nuance expected of a senior news analyst.

**Top 5 News Stories - July 11th, 2026**

**(Note: All times are GMT)**


**1.  Global Maritime Conflict Escalates – Southern Atlantic Treaty Forces Mobilized (Ongoing)**
   * **Source:** Reuters & BBC
   * **Summary**: The simmering tensions in the South Atlantic have boiled over with confirmed reports of a significant naval clash between the Federated Republic of Azmar and a coalition led by the Argento-Brazilian Defense Pact (ABDP) near the disputed Clarion Ridge. Initial reports suggested Azmari forces targeted ABDP supply convoys believed to be resupplying territorial claims, though the ABDP has vehemently denied accusations of provocations and maintained that Azmar initiated hostilities. Multiple civilian vessels have been caught in the crossfire, leading to a growing humanitarian crisis.  The United Nations Security Council convened an emergency session but was quickly deadlocked as permanent member nations remained divided, with France strongly advocating for intervention under Chapter VII authority.
   * **Impact Analysis:** This escalation marks a sharp deterioration of global stability. The Clarion Ridge dispute has long been considered a low-level flashpoint, but this confrontation reveals a worrying increase in aggressive posturing by resource-rich nations and underscores the vulnerability of international shipping lanes; potentially disrupting trade routes vital to the EU economy.  The mobilization of the Southern Atlantic Treaty Forces – a largely dormant military alliance between Argentina, Brazil, and several smaller South American states – adds significant strategic weight and raises concerns about a wider regional conflict. The most immediate concern is the disruption to the lucrative deep-sea mining industry concentrated around Clarion Ridge and the increasing risk of civilian casualties.  A UN intervention, while desired by many, hangs in the balance due to geopolitical tensions. *Probability of further escalation: 65%*



**2.  Neo-Luddite Uprising in Neo-Kyoto – Autonomous Worker Actions Continue (Ongoing)**
   * **Source:** CNN & AP
   * **Summary**: The widespread autonomous worker protests, dubbed the “Zero State” movement, are deepening across Japan and rapidly spreading to other heavily industrialized nations, particularly within the Pan-Asian Economic Union (PAEU).  Initially focused on resisting increasingly sophisticated algorithmic management in factories and logistics, the movement has shifted its demands towards a complete cessation of automation – advocating for a return to traditional human labor. Violent clashes between protestors and autonomous enforcement units – “Drone Guardians” - are escalating nightly across Neo-Kyoto, with reports of significant infrastructure damage due to targeted attacks on automated systems. Negotiated dialogue has collapsed, leading to an uncertain future.
   * **Impact Analysis:** The spread of the "Zero State" movement represents a fundamental challenge to the global economic model predicated on automation and efficiency. While initially dismissed as fringe radicalism, its increasing momentum highlights legitimate anxieties concerning job displacement, algorithmic bias, and the erosion of worker autonomy. The actions in Neo-Kyoto could trigger a broader societal breakdown if not contained.  The PAEU governments are utilizing increasingly stringent counter-protests, including cyberattacks and restrictions on movement – potentially triggering a major trade dispute or wider unrest within the region. *Probability of wider spread: 70%*



**3.  Lunar Ice Breakthrough Confirmed - Sino-European Consortium Announces Major Discovery (8:00 GMT)**
   * **Source:** BBC & Al Jazeera
   * **Summary**: The Sino-European Lunar Exploration Consortium (SELEC) announced today the confirmed discovery of a significantly larger and purer source of water ice than previously anticipated on the far side of the Moon.  Preliminary data suggests concentrations exceeding initial estimates by almost 300%, offering potentially limitless resources for future lunar settlements and propellant production. The consortium is already planning accelerated development of robotic mining infrastructure, utilizing advanced 3D-printing techniques to construct initial processing facilities.
   * **Impact Analysis:** This discovery dramatically shifts the geopolitical landscape surrounding space exploration.  It immediately elevates China and Europe’s strategic positions as dominant players in lunar resource development – potentially triggering a new "Space Race" that could exacerbate existing tensions with the United States and other nations involved in independent lunar programs. The economic implications are enormous, offering prospects for sustainable rocket fuel production and supporting nascent lunar economies. However, concerns about equitable access to these resources and responsible exploitation of the lunar environment remain paramount. *Probability of geopolitical disruption: 50%*



**4.  Synthetic Food Crisis Deepens – Global Supply Chain Collapses (12:00 GMT)**
    * **Source**: Reuters & AP
    * **Summary:** A perfect storm of factors – including a prolonged avian influenza outbreak impacting vertical farming operations and a devastating cyberattack targeting the central distribution networks utilized by SynFoods Corp – has triggered a severe global shortage of synthetic protein sources. Prices have skyrocketed, creating food security anxieties across densely populated regions reliant on SynFood production. Governments are scrambling to implement temporary rationing measures.
    * **Impact Analysis**: This represents an exacerbation of a pre-existing trend highlighting the fragility of our reliance on heavily centralized, technologically dependent food systems. The collapse further underlines the risks associated with "peak protein" – a period where demand vastly outstrips sustainable production methods.  The situation poses serious economic and humanitarian concerns, particularly impacting vulnerable populations in developing nations. The event is likely to spur significant investment in localized food security initiatives and potentially accelerate a shift towards more traditional farming practices. *Probability of widespread unrest: 40%*



**5.  AI Ethics Tribunal Issues Landmark Ruling – "Sentience Threshold" Defined (16:00 GMT)**
   * **Source:** CNN & Al Jazeera
   * **Summary**: The International AI Ethics Tribunal (IAET) delivered a landmark ruling today formally defining the “sentience threshold” for legal consideration of Artificial General Intelligence.  Following years of debate, the Tribunal established that an AGI must demonstrate genuine self-awareness - exhibiting independent thought patterns beyond mere sophisticated mimicry – to be afforded fundamental rights and protections under international law. The decision has ignited fierce controversy within the tech industry.
   * **Impact Analysis:** This ruling represents a pivotal moment in the ongoing discussion regarding AI governance. It raises fundamental questions about consciousness, personhood, and our ethical obligations towards increasingly advanced artificial intelligence systems.  The Tribunal's definition creates significant legal challenges for developing AGI technology – forcing developers to prioritize genuine sentience over purely algorithmic performance. The decision will undoubtedly fuel a new wave of philosophical and ethical debates surrounding the long-term implications of AI’s development. *Probability of legal challenges & technological slowdown: 80%*



---

**Disclaimer:** As a simulated news analyst, this report is entirely fictional and created for the purpose of fulfilling your prompt. It's based on extrapolations and informed speculation about potential future developments.  Actual events will undoubtedly differ.

In [11]:
# ========== Argos Translate：安装英→阿语包并做一次翻译冒烟测试 ==========

# 导入 Argos 的包管理与翻译 API
import argostranslate.package
import argostranslate.translate

# 更新可安装语言包索引（需联网；通常执行一次即可）
argostranslate.package.update_package_index()
# 取出所有可下载的语言包列表
available_packages = argostranslate.package.get_available_packages()
# 筛选：英语(en) → 阿拉伯语(ar)
package_to_install = next(
    filter(
        lambda x: x.from_code == "en" and x.to_code == "ar",
        available_packages
    )
)
# 下载并安装该语言包到本机
argostranslate.package.install_from_path(package_to_install.download())

# 现在翻译一句 Hello world，验证链路是否通
translated = argostranslate.translate.translate("Hello world", "en", "ar")
print(translated)


2026-07-13 02:47:59 WARNING: Language en package default expects mwt, which has been added


مرحبا


In [12]:
# ========== 新闻文章分析：newspaper3k 抓取 + 本地 Ollama 摘要/影响 ==========

from openai import OpenAI
from newspaper import Article

# ----------------------------
# 1. AI 提示词（发给模型的英文保持原样）
# ----------------------------
system_prompt = "You are a senior news analyst. Read the provided article and produce a professional summary and impact analysis."

user_prompt_template = """
Analyze the following news article text.

Provide:
- A concise 3-sentence summary of the key facts.
- A brief analysis of its potential political, economic, or social impact.

--- Article ---
{article_text}
--- End of Article ---

Format your response exactly like this:
Summary: [Your summary]
Impact: [Your impact analysis]
"""

# ----------------------------
# 2. 使用 newspaper3k 从 URL 获取并清理文章
# ----------------------------
def fetch_article_text(url):
    try:
        # Article：newspaper3k 的核心对象，负责下载与解析
        article = Article(url)
        # 下载网页 HTML
        article.download()
        # 解析标题、正文等字段
        article.parse()

        # 检查我们是否确实获得了足够正文（太短多半是视频/付费墙/需 JS）
        if len(article.text) < 100:
            return "Error: The article text is too short. The page might be a video, paywalled, or require JavaScript."

        # 拼成「Title + 正文」字符串，交给后续 prompt
        full_text = f"Title: {article.title}\n\n{article.text}"
        return full_text
    except Exception as e:
        # 任何下载/解析异常都转成 Error 前缀字符串，便于主流程分支
        return f"Error fetching the article: {e}"

# ----------------------------
# 3. 为 Ollama 准备 messages
# ----------------------------
def messages_for_article(article_text):
    # 截断以避免超出 Gemma 的上下文窗口（约 6000 个字符是安全的）
    truncated = article_text[:6000]
    return [
        {"role": "system", "content": system_prompt},
        # format：把正文填进 user_prompt_template 的 {article_text}
        {"role": "user", "content": user_prompt_template.format(article_text=truncated)}
    ]

# ----------------------------
# 4. 主程序：交互输入 URL → 抓取 → 本地模型分析 → Markdown 展示
# ----------------------------
if __name__ == "__main__":
    # 询问用户粘贴新闻 URL（strip 去掉首尾空白）
    url = input("Paste the news article URL: ").strip()

    print("\n📰 Fetching article...")
    article_content = fetch_article_text(url)

    # 约定：错误信息以 "Error" 开头
    if article_content.startswith("Error"):
        print(f"\n❌ {article_content}")
    else:
        print("🤖 Generating summary & impact analysis...\n")

        # 连接到本地 Ollama
        ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

        # 得到回应
        response = ollama.chat.completions.create(
            model="gemma3:4b",
            messages=messages_for_article(article_content)
        )

        # 打印分隔线，并用 Markdown 渲染模型输出
        print("=" * 50)
        display(Markdown(response.choices[0].message.content))
        print("=" * 50)



📰 Fetching article...
🤖 Generating summary & impact analysis...



Okay, here's an analysis of the provided news article, formatted as requested:

**Summary:** On July 11, 2026, the United States launched a third round of strikes against Iran following an attack on a Cypriot-flagged container ship in the Strait of Hormuz. The incident began with Iranian forces firing a warning shot at a vessel attempting to use an unauthorized route and subsequently closed the strait, leading to reciprocal actions including attacks on Jordanian military installations. Mediators are currently working through a tentative proposal drafted by Oman to ensure safe passage for ships in the strategically vital waterway, though US demands regarding ship safety remain a key obstacle to diplomatic progress.

**Impact:** This escalation presents significant geopolitical risk and has the potential for substantial economic fallout. Militarily, the strikes sharply raise the possibility of further conflict between the US and Iran, potentially drawing in regional allies like the UAE and Qatar who responded with defensive measures. Economically, the Strait of Hormuz is a critical chokepoint for global oil shipments; any prolonged disruption due to continued tensions or military action could trigger an immediate spike in energy prices and destabilize international markets. Socially, heightened tensions further complicate diplomatic efforts and contribute to an already volatile regional landscape, with potential ramifications for maritime security and trade routes.

In [13]:
"""
News Article Analyzer with Arabic Translation
----------------------------------------------
Fetches a news article, generates a summary + impact analysis with a local
Ollama model (gemma3:4b), then translates both into Arabic with Argos Translate.

Requirements:
    pip install openai beautifulsoup4 requests argostranslate
    Ollama running locally:  ollama pull gemma3:4b
"""

# ========== 导入：HTTP 抓取、HTML 解析、本地 LLM、离线翻译 ==========

import json
import re
import textwrap

# requests：发 HTTP GET 拉网页
import requests
# BeautifulSoup：从 HTML 里抽出可读正文
from bs4 import BeautifulSoup
# OpenAI 客户端：这里用来打本地 Ollama 的 /v1
from openai import OpenAI

# Argos Translate：本地神经机器翻译（可英→阿）
import argostranslate.package
import argostranslate.translate

# ----------------------------
# 1. 配置：端点、模型、语言、截断长度等集中放这里
# ----------------------------
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "gemma3:4b"
FROM_LANG = "en"
TO_LANG = "ar"
MAX_ARTICLE_CHARS = 6000      # how much of the article to send to the model
WRAP_WIDTH = 120

# 浏览器风格 User-Agent：降低部分站点直接拒绝脚本访问的概率
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    )
}

# ----------------------------
# 2. AI 提示（要求 JSON，而不是自由文本；英文指令保持原样）
# ----------------------------
SYSTEM_PROMPT = (
    "You are a senior news analyst. Read the provided article and produce a "
    "professional summary and impact analysis. Respond ONLY with valid JSON."
)

USER_PROMPT_TEMPLATE = """Analyze the following news article.

Return a JSON object with exactly two string fields:
- "summary": a concise 3-sentence summary of the key facts.
- "impact": a brief analysis of the potential political, economic, or social impact.

Do not include any text outside the JSON object.

--- Article ---
{article_text}
--- End of Article ---
"""


# ----------------------------
# 3. 获取并清理文章（requests + BeautifulSoup）
# ----------------------------
def fetch_article_text(url):
    """Return 'Title\\n\\ntext' for the page, or an 'Error: ...' string on failure."""
    try:
        # GET 网页；timeout 避免一直挂起
        response = requests.get(url, headers=HEADERS, timeout=20)
        # 非 2xx 直接抛异常，进入 except
        response.raise_for_status()
    except Exception as e:
        return f"Error fetching the article: {e}"

    # 用 html.parser 解析字节内容
    soup = BeautifulSoup(response.content, "html.parser")
    # 尽量取出 <title>；没有就用占位文案
    title = (
        soup.title.string.strip()
        if soup.title and soup.title.string
        else "No title found"
    )

    if soup.body:
        # 去掉脚本/样式/导航等噪音标签，再抽取正文文本
        for tag in soup.body(["script", "style", "img", "input", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    # 正文过短：多半抓取失败或页面非文章
    if len(text) < 100:
        return (
            "Error: The article text is too short. The page might be a video, "
            "paywalled, or require JavaScript."
        )

    return f"Title: {title}\n\n{text}"


# ----------------------------
# 4. 模型调用 + 稳健解析（JSON / 围栏 / 标签文本三级兜底）
# ----------------------------
def build_messages(article_text):
    # 截断正文，避免撑爆本地小模型上下文
    truncated = article_text[:MAX_ARTICLE_CHARS]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(article_text=truncated)},
    ]


def _strip_code_fences(text):
    """Remove ```json ... ``` or ``` ... ``` wrappers if the model added them."""
    text = text.strip()
    # 去掉开头的 ```json / ``` 围栏
    text = re.sub(r"^```(?:json)?\s*", "", text)
    # 去掉结尾的 ```
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def parse_result(raw):
    """
    Pull 'summary' and 'impact' out of the model output as reliably as possible.
    Returns (summary, impact); either may be None.
    """
    cleaned = _strip_code_fences(raw)

    # 1) 整个响应就是 JSON 对象
    try:
        data = json.loads(cleaned)
        return data.get("summary"), data.get("impact")
    except (json.JSONDecodeError, AttributeError):
        pass

    # 2) JSON 对象嵌在其他文字中间：用正则捞出 {...}
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            return data.get("summary"), data.get("impact")
        except json.JSONDecodeError:
            pass

    # 3) 后备：基于 Summary:/Impact: 标签的纯文本（允许 **, #, 空白, 多行）
    summary_m = re.search(
        r"Summary\s*:?\**\s*(.+?)(?=\n?\**#*\s*Impact\b|$)",
        cleaned, re.IGNORECASE | re.DOTALL,
    )
    impact_m = re.search(r"Impact\s*:?\**\s*(.+)", cleaned, re.IGNORECASE | re.DOTALL)
    summary = summary_m.group(1).strip(" *#\n") if summary_m else None
    impact = impact_m.group(1).strip(" *#\n") if impact_m else None
    return summary, impact


def analyze_article(article_text):
    # 创建指向本地 Ollama 的客户端
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
    # 公共参数：模型、messages、较低 temperature 让输出更稳
    kwargs = dict(model=MODEL, messages=build_messages(article_text), temperature=0.3)
    try:
        # 当版本支持 JSON 时，向 Ollama 询问 JSON（response_format）
        response = client.chat.completions.create(
            **kwargs, response_format={"type": "json_object"}
        )
    except Exception:
        # 较旧的 Ollama 版本可能会拒绝 response_format；回退为普通调用
        response = client.chat.completions.create(**kwargs)
    raw = response.choices[0].message.content
    # 同时返回原始文本与解析后的 (summary, impact)
    return raw, parse_result(raw)


# ----------------------------
# 5. Argos 翻译：检查/安装语言包，再做英→阿
# ----------------------------
def _translation_available(from_lang, to_lang):
    # 列出已安装语言，看是否已有 from→to 的翻译器
    langs = argostranslate.translate.get_installed_languages()
    from_l = next((l for l in langs if l.code == from_lang), None)
    to_l = next((l for l in langs if l.code == to_lang), None)
    if not from_l or not to_l:
        return False
    try:
        return from_l.get_translation(to_l) is not None
    except Exception:
        return False


def ensure_translation_installed(from_lang=FROM_LANG, to_lang=TO_LANG):
    """Install the from->to Argos package once, if it isn't already present."""
    if _translation_available(from_lang, to_lang):
        return True

    print("📦 Downloading translation package (English → Arabic)...")
    argostranslate.package.update_package_index()
    available = argostranslate.package.get_available_packages()
    pkg = next(
        (p for p in available if p.from_code == from_lang and p.to_code == to_lang),
        None,
    )
    if pkg is None:
        print("⚠️ English → Arabic package not found.")
        return False

    argostranslate.package.install_from_path(pkg.download())
    print("✅ Translation package installed.")
    return True


def translate_text(text, from_lang=FROM_LANG, to_lang=TO_LANG):
    try:
        # Argos 离线翻译：源语言码 → 目标语言码
        return argostranslate.translate.translate(text, from_lang, to_lang)
    except Exception as e:
        return f"[Translation error: {e}]"


# ----------------------------
# 6. 输出助手：自动换行 + 英/阿双语区块
# ----------------------------
def wrap_text(text, width=WRAP_WIDTH):
    if not text:
        return ""
    # 按空行分段，再对每段 fill 到固定宽度，避免终端一行过长
    paragraphs = text.split("\n\n")
    return "\n\n".join(
        textwrap.fill(p, width=width, break_long_words=False) for p in paragraphs
    )


def print_section(heading, english):
    print(f"\n## **{heading}**")
    if english:
        print(f"🇬🇧 **English:**\n{wrap_text(english)}")
        arabic = translate_text(english)
        print(f"\n🇸🇦 **Arabic Translation (الترجمة العربية):**\n{wrap_text(arabic)}")
    else:
        print(f"No {heading.lower()} found.")


# ----------------------------
# 7. 主流程：装翻译包 → 抓文 → 分析 → 双语打印
# ----------------------------
def main():
    # 确保 en→ar 包可用（没有就下载）
    ensure_translation_installed()

    url = input("Paste the news article URL: ").strip()

    print("\n📰 Fetching article...")
    article = fetch_article_text(url)
    if article.startswith("Error"):
        print(f"\n❌ {article}")
        return

    print("🤖 Generating summary & impact analysis...\n")
    raw, (summary, impact) = analyze_article(article)

    # 安全网：如果没有解析任何字段，则展示原始输出而不是丢掉
    if not summary and not impact:
        print("\n⚠️ Could not parse structured fields. Raw model output:\n")
        print(wrap_text(raw))
        return

    print("\n" + "=" * 50)
    print("# 📰 Article Analysis")
    print_section("Summary", summary)
    print_section("Impact", impact)
    print("=" * 50)


if __name__ == "__main__":
    main()



📰 Fetching article...
🤖 Generating summary & impact analysis...


# 📰 Article Analysis

## **Summary**
🇬🇧 **English:**
Following an attack on a merchant ship in the Strait of Hormuz, the US military launched its third round of strikes
against Iran, targeting approximately 140 Iranian military sites including missile and drone facilities. This escalation
was prompted by Iran’s declaration of the waterway closed and alleged ‘outside interference’ with unauthorized shipping
routes, leading to retaliatory attacks on Jordan's Prince Hassan Air Base. Regional tensions have dramatically increased
with reports of explosions along Iran’s southern coast and intercepted aerial targets across multiple Gulf nations.

🇸🇦 **Arabic Translation (الترجمة العربية):**
وعقب هجوم على سفينة تجار في مضيق هورموز، شن الجيش الأمريكي جولته الثالثة من الضربات ضد إيران، واستهدف نحو 140 موقعا
عسكريا إيرانيا، بما في ذلك مرافق القذائف والطائرات بدون طيار. وقد أدى هذا التصعيد إلى إعلان إيران عن المجرى المائي
المغلق وا